## This notebook launches the LatissCWFSAlign script, meant for the scriptQueue, but via a notebook.
##### This calculates focus offsets for the hexapod (and also decentering corrections)

In [1]:
import sys
import asyncio
import time

import numpy as np
import logging 
import yaml
import matplotlib.pyplot as plt

from lsst.ts import salobj
from lsst.ts.externalscripts.auxtel.latiss_cwfs_align import LatissCWFSAlign

from lsst.ts.idl.enums.Script import ScriptState

In [2]:
stream_handler = logging.StreamHandler(sys.stdout)
# if you want logging
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG

# turn off logging for matplotlib
mpl_logger = logging.getLogger('matplotlib')
mpl_logger.setLevel(logging.WARNING)


In [3]:
script = LatissCWFSAlign(index=1, remotes=True)  # this essentially calls the init method
# script = LatissCWFSAlign(index=1, remotes=True, index=os.getuid())
# make sure all remotes etc are running
#await script.start_task

DEBUG:Script.ATCS:atmcs: Adding all resources.
atmcs: Adding all resources.
DEBUG:Script.ATCS:atptg: Adding all resources.
atptg: Adding all resources.
DEBUG:Script.ATCS:ataos: Adding all resources.
ataos: Adding all resources.
DEBUG:Script.ATCS:atpneumatics: Adding all resources.
atpneumatics: Adding all resources.
DEBUG:Script.ATCS:athexapod: Adding all resources.
athexapod: Adding all resources.
DEBUG:Script.ATCS:atdome: Adding all resources.
atdome: Adding all resources.
DEBUG:Script.ATCS:atdometrajectory: Adding all resources.
atdometrajectory: Adding all resources.
DEBUG:Script.LATISS:atcamera: Adding all resources.
atcamera: Adding all resources.
DEBUG:Script.LATISS:atspectrograph: Adding all resources.
atspectrograph: Adding all resources.
DEBUG:Script.LATISS:atheaderservice: Adding all resources.
atheaderservice: Adding all resources.
DEBUG:Script.LATISS:atarchiver: Adding all resources.
atarchiver: Adding all resources.
INFO:Script:latiss_cwfs_align initialized!
latiss_cwfs_a

In [ ]:
# It looks like this command will do the 06mm x aor y offsets
# await atcs.rem.ataos.cmd_offset.set_start(x=0.6)


In [ ]:
# set wrap strategy
# this is required until the ATPtg is updated to not configure the mount for maximum time on target
#script.atcs.rem.atptg.cmd_raDecTarget.set(azWrapStrategy=1)  # 1 does not unwrap, 0 unwraps

## Emulate how the scriptQueue launches scripts
##### Start here if re-running the script after a correction

In [27]:
configuration = yaml.safe_dump({"filter": 'RG610', 
                                "grating": 'empty_1',
                                "exposure_time": 20,
                                "dataPath": '/project/shared/auxTel/rerun/quickLook'})
print(configuration)

dataPath: /project/shared/auxTel/rerun/quickLook
exposure_time: 20
filter: RG610
grating: empty_1



In [28]:
# Set script state to UNCONFIGURED
# this is required to run the script a 2nd time but otherwise is a no-op
script.set_state(ScriptState.UNCONFIGURED)
# Configure the script, which puts the ScriptState to CONFIGURED
config_data = script.cmd_configure.DataType()
config_data.config = configuration
await script.do_configure(config_data)

INFO:Script:Using binning factor of 1
Using binning factor of 1


In [ ]:
# ATAOS must be on and corrections enabled, do as follows if required
#await script.atcs.rem.ataos.cmd_enableCorrection.set_start(m1=True, hexapod=True, atspectrograph=True)

In [ ]:
# Target must be on the detector
# Can use this command to slew to target if required
# await script.attcs.slew_object('96 Cet')

In [29]:
# Run this script while taking new data
script.intra_visit_id = None
script.extra_visit_id = None
script.short_timeout = 10
results = await script.arun()

DEBUG:Script:CWFS iteration 1 starting...
CWFS iteration 1 starting...
Intra/Extra images not taken. Running take image sequence.
DEBUG:Script:Moving to intra-focal position
Moving to intra-focal position
DEBUG:Script.LATISS:imagetype: ENGTEST, skip TCS synchronization.
imagetype: ENGTEST, skip TCS synchronization.
DEBUG:Script.LATISS:ENGTEST 0001 - 0001
ENGTEST 0001 - 0001
DEBUG:Script:Moving to extra-focal position
Moving to extra-focal position
DEBUG:Script:Taking extra-focal image
Taking extra-focal image
DEBUG:Script.LATISS:imagetype: ENGTEST, skip TCS synchronization.
imagetype: ENGTEST, skip TCS synchronization.
DEBUG:Script.LATISS:ENGTEST 0001 - 0001
ENGTEST 0001 - 0001
INFO:Script:intraImage expId for target: 2021060800385
intraImage expId for target: 2021060800385
INFO:Script:extraImage expId for target: 2021060800386
extraImage expId for target: 2021060800386
INFO:Script:angle used in cwfs algorithm is -91.00869192917992
angle used in cwfs algorithm is -91.00869192917992
DEB

In [85]:
from lsst.ts.observatory.control.utils import RotType
await script.atcs.slew_object('HD 200502', rot_type=RotType.Parallactic)

DEBUG:urllib3.connectionpool:Starting new HTTP connection (1): simbad.u-strasbg.fr:80
Starting new HTTP connection (1): simbad.u-strasbg.fr:80
DEBUG:urllib3.connectionpool:http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
http://simbad.u-strasbg.fr:80 "POST /simbad/sim-script HTTP/1.1" 200 None
INFO:Script.ATCS:Slewing to HD 200502: 21 05 23.2369 -45 14 30.469
Slewing to HD 200502: 21 05 23.2369 -45 14 30.469
DEBUG:Script.ATCS:Setting rotator position with respect to parallactic angle to 0.0 deg.
Setting rotator position with respect to parallactic angle to 0.0 deg.
DEBUG:Script.ATCS:Parallactic angle: -81.9214668896843 | Sky Angle: 8.078533110315703
Parallactic angle: -81.9214668896843 | Sky Angle: 8.078533110315703
DEBUG:Script.ATCS:Sending command
Sending command
DEBUG:Script.ATCS:Stop tracking.
Stop tracking.
target python read queue is filling: 36 of 100 elements
DEBUG:Script.ATCS:Tracking state: <AtMountState.TRACKINGENABLED: 9>
Tracking state: <AtMountSta

In [88]:
await script.latiss.take_object(2.0, 1, filter='RG610', grating='empty_1')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started', 'completed'].
Received corrections: ['started', 'completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:No correction seen in the last 5.0 seconds. Determining order of last corrections.
No correction seen in the last 5.0 seconds. Determining order of last corrections.
DEBUG:Script.ATCS:Last correction co

array([2021060800429])

In [87]:
await script.atcs.offset_xy(y=-50, x=-10, relative=True)

DEBUG:Script.ATCS:Calculating x/y offset: -10/-50 
Calculating x/y offset: -10/-50 
DEBUG:Script.ATCS:Applying Az/El offset: 50.011807659134405/-9.940779379190811 
Applying Az/El offset: 50.011807659134405/-9.940779379190811 
DEBUG:Script.ATCS:Telescope not in position.
Telescope not in position.
DEBUG:Script.ATCS:All axes in position.
All axes in position.
DEBUG:Script.ATCS:Waiting for telescope to settle.
Waiting for telescope to settle.
DEBUG:Script.ATCS:Done
Done


In [149]:
offset = {
            "m1": 0.0,
            "m2": 0.0,
            "x": 0.0,
            "y": 2.0,
            "z": 0.0,
            "u": 0.0,
            "v": 0.0,
        }
        
await script.atcs.rem.ataos.cmd_offset.set_start(
            **offset, timeout=script.short_timeout)

In [150]:
await script.latiss.take_object(2.0, 1, filter='RG610', grating='empty_1')

DEBUG:Script.LATISS:Generating group_id
Generating group_id
DEBUG:Script.LATISS:imagetype: OBJECT, wait for TCS to be ready.
imagetype: OBJECT, wait for TCS to be ready.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:Received corrections: ['started', 'completed'].
Received corrections: ['started', 'completed'].
DEBUG:Script.ATCS:Ready to take data:: atmcs=True, ataos=False, atdome=True.
Ready to take data:: atmcs=True, ataos=False, atdome=True.
DEBUG:Script.ATCS:atspectrograph correction running. Trying to determine state of the corrections.
atspectrograph correction running. Trying to determine state of the corrections.
DEBUG:Script.ATCS:No correction seen in the last 5.0 seconds. Determining order of last corrections.
No correction seen in the last 5.0 seconds. Determining order of last corrections.
DEBUG:Script.ATCS:Last correction co

array([2021060800472])

In [151]:
# Run this script while taking new data
script.intra_visit_id = None
script.extra_visit_id = None

In [152]:
await script.take_intra_extra()

DEBUG:Script:Moving to intra-focal position
Moving to intra-focal position
DEBUG:Script.LATISS:imagetype: ENGTEST, skip TCS synchronization.
imagetype: ENGTEST, skip TCS synchronization.
DEBUG:Script.LATISS:ENGTEST 0001 - 0001
ENGTEST 0001 - 0001
DEBUG:Script:Moving to extra-focal position
Moving to extra-focal position
DEBUG:Script:Taking extra-focal image
Taking extra-focal image
DEBUG:Script.LATISS:imagetype: ENGTEST, skip TCS synchronization.
imagetype: ENGTEST, skip TCS synchronization.
DEBUG:Script.LATISS:ENGTEST 0001 - 0001
ENGTEST 0001 - 0001
INFO:Script:intraImage expId for target: 2021060800473
intraImage expId for target: 2021060800473
INFO:Script:extraImage expId for target: 2021060800474
extraImage expId for target: 2021060800474
INFO:Script:angle used in cwfs algorithm is -98.70345449512169
angle used in cwfs algorithm is -98.70345449512169
DEBUG:Script:Moving hexapod back to zero offset (in-focus) position
Moving hexapod back to zero offset (in-focus) position
logMessage

In [ ]:
# show donuts and centroids
fig1 = plt.figure(1, figsize=(12,8))
ax11 = fig1.add_subplot(121)
ax11.set_title(f"intra visitID - {script.intra_visit_id}")
ax11.imshow(script.I1[0].image0)
ax11.contour(script.algo.pMask) 
ax12 = fig1.add_subplot(122)
ax12.set_title(f"extra visitID - {script.extra_visit_id}")
ax12.imshow(script.I2[0].image0)
ax12.contour(script.algo.pMask) 

In [ ]:
# Apply calculated focus offset
#calculated_hexapod_focus_offset = results['hex_offset'][2]
#print(f'Applying focus offset of {calculated_hexapod_focus_offset}')
#await script.atcs.rem.ataos.cmd_offset(z=calculated_hexapod_focus_offset)

# Stop here unless a re-reduction of the doughnuts is required.

#### If you want to re-reduce data then use the below cells

In [ ]:
# Show which files/parameters were taken in the sequence above
print(f'intra_visit_id is {script.intra_visit_id}')
print(f'extra_visit_id is {script.extra_visit_id}')
print(f'angle is {script.angle}')

In [ ]:
# If desired then different filenames can be manually input here
#script.intra_visit_id=2021011900169 
#script.extra_visit_id=2021011900170 
#script.angle=-91.56748047249727

In [ ]:
# reruns reduction part only. ALL 3 fields above must be set! 
rerun_results = await script.run_cwfs()